# 预期ST因子测试

因子定义：连续两年净利润为负（老规则）

策略逻辑：
1. 每天从全市场筛选总市值最小的500只股票（尾部500）
2. 在尾部500中，筛选连续两年净利润为负的股票
3. 等权持有所有满足条件的股票
4. 每天调仓

In [10]:
from bigmodule import M, I
import dai
import pandas as pd


def m5_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


def m5_before_trading_start_bigquant_run(context, data):
    pass


def m5_handle_tick_bigquant_run(context, tick):
    pass


def m5_handle_data_bigquant_run(context, data):
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    target_instruments = set(today_df["instrument"])
    holding_instruments = set(context.get_account_positions().keys())

    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)

    for i, x in today_df.iterrows():
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        context.order_target_percent(x.instrument, position)


def m5_handle_trade_bigquant_run(context, trade):
    pass


def m5_handle_order_bigquant_run(context, order):
    pass


def m5_after_trading_bigquant_run(context, data):
    pass


# ========== 数据准备 ==========
# 预期ST因子：连续两个财政年度净利润为负（老规则）
# 使用 cn_stock_financial_lf_shift 表获取精确的财年净利润

stock_sql = """
WITH 
-- 获取最近两个财年的净利润 (shift=0 当年, shift=1 上一年)
profit_lf AS (
    SELECT
        date,
        instrument,
        MAX(CASE WHEN shift = 0 THEN net_profit_lf END) AS net_profit_y0,
        MAX(CASE WHEN shift = 1 THEN net_profit_lf END) AS net_profit_y1
    FROM cn_stock_financial_lf_shift
    WHERE shift IN (0, 1)
    GROUP BY date, instrument
),
-- 基础股票池：非ST、非停牌、非北交所、上市满1年
base AS (
    SELECT
        a.date,
        a.instrument,
        a.total_market_cap,
        b.net_profit_y0,
        b.net_profit_y1
    FROM cn_stock_prefactors_community a
    JOIN profit_lf b USING (date, instrument)
    WHERE
        a.st_status = 0
        AND a.suspended = 0
        AND a.is_bz50 = 0
        AND a.list_days > 365
    QUALIFY
        ROW_NUMBER() OVER (PARTITION BY a.date ORDER BY a.total_market_cap ASC) <= 500
),
-- 筛选：连续两个财年净利润为负
filtered AS (
    SELECT * FROM base
    WHERE net_profit_y0 < 0 AND net_profit_y1 < 0
)
SELECT
    date,
    instrument,
    total_market_cap,
    net_profit_y0,
    net_profit_y1,
    -total_market_cap AS score,
    1.0 / c_sum(1) AS position
FROM filtered
ORDER BY date, instrument
"""

print("正在查询数据...")
filtered_df = dai.query(stock_sql, filters={"date": ["2020-01-01", "2026-12-31"]}).df()
print(f"满足预期ST条件的记录数：{len(filtered_df)}")
print(f"每日平均持仓数量：{filtered_df.groupby('date')['instrument'].count().mean():.1f}")

stock_data_ds = dai.DataSource.write_bdb(filtered_df)

# ========== 回测 ==========
start_date = '2021-01-01'
end_date = '2026-04-07'

m5 = M.bigtrader.v30(
    data=stock_data_ds,
    start_date=start_date,
    end_date=end_date,
    initialize=m5_initialize_bigquant_run,
    before_trading_start=m5_before_trading_start_bigquant_run,
    handle_tick=m5_handle_tick_bigquant_run,
    handle_data=m5_handle_data_bigquant_run,
    handle_trade=m5_handle_trade_bigquant_run,
    handle_order=m5_handle_order_bigquant_run,
    after_trading=m5_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="daily",
    product_type="股票",
    rebalance_period_type="交易日",
    rebalance_period_days="1",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="标准模式",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="open",
    order_price_field_sell="open",
    benchmark="沪深300指数",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="m5"
)

正在查询数据...
满足预期ST条件的记录数：8566
每日平均持仓数量：16.5
[2026-04-08 11:26:02] [info     ] bigtrader.v30 开始运行 ..
[2026-04-08 11:26:02] [info     ] read input 'data' ..
[2026-04-08 11:26:02] [info     ] 2021-01-01, 2026-04-07, , equity, instruments=1986
[2026-04-08 11:26:02] [info     ] bigtrader module V2.1.0
[2026-04-08 11:26:02] [info     ] bigtrader engine v0.1.0.post9+g6d7300d 2026-02-10
[2026-04-08 11:26:33] [info     ] backtest done, raw_perf_ds:dai.DataSource("_c26bed2396a34995bf5f4d889e0061c7")


[2026-04-08 11:26:35] [info     ] bigtrader.v30 运行完成 [33.072s].


In [15]:
# ========== 导出交易记录CSV ==========
raw_perf = m5.raw_perf.read()
trades_list = [t for txns in raw_perf['transactions'] if txns for t in txns]
trades_df = pd.DataFrame(trades_list)

# 获取所有交易涉及的股票
instruments = trades_df['symbol'].unique().tolist()

# 查询股票名称和行业（取最新数据）
info_sql = """
SELECT DISTINCT ON (instrument)
    instrument,
    name
FROM cn_stock_prefactors_community
ORDER BY instrument, date DESC
"""
stock_info = dai.query(info_sql, filters={"date": ["2020-01-01", "2030-01-01"]}).df()
stock_info_dict = stock_info.set_index('instrument')['name'].to_dict()

industry_sql = """
SELECT DISTINCT ON (instrument)
    instrument,
    industry_level1_name,
    industry_level2_name
FROM cn_stock_industry_component
ORDER BY instrument, date DESC
"""
industry_info = dai.query(industry_sql, filters={"date": ["2020-01-01", "2030-01-01"]}).df()
industry_dict = industry_info.set_index('instrument')[['industry_level1_name', 'industry_level2_name']].to_dict('index')

output_records = []
holdings = {}

for idx, row in trades_df.iterrows():
    instrument = row['symbol']
    dt = pd.to_datetime(row['dt']).strftime('%Y-%m-%d')
    amount = row['amount']
    price = row['price']
    
    if amount > 0:
        holdings[instrument] = {'buy_date': dt, 'buy_price': price}
    elif amount < 0 and instrument in holdings:
        buy_info = holdings.pop(instrument)
        pnl = (price - buy_info['buy_price']) / buy_info['buy_price']
        ind = industry_dict.get(instrument, {})
        output_records.append({
            '股票代码': instrument.split('.')[0],
            '股票名': stock_info_dict.get(instrument, ''),
            '行业分类': ind.get('industry_level1_name', ''),
            '二级行业': ind.get('industry_level2_name', ''),
            '买入日期': buy_info['buy_date'],
            '卖出日期': dt,
            '买入价格(前复权)': round(buy_info['buy_price'], 2),
            '卖出价格(前复权)': round(price, 2),
            '涨幅': round(pnl, 4)
        })

output_df = pd.DataFrame(output_records)
output_df = output_df.sort_values('卖出日期', ascending=False)

output_path = './预期ST_bigquant交易记录.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"交易记录已保存到：{output_path}")
print(f"共 {len(output_df)} 条交易记录")
output_df.head(20)

交易记录已保存到：./预期ST_bigquant交易记录.csv
共 6776 条交易记录


,股票代码,股票名,行业分类,二级行业,买入日期,卖出日期,买入价格(前复权),卖出价格(前复权),涨幅
6775,688216,气派科技,电子,半导体,2026-04-03,2026-04-07,27.22,26.46,-0.0279
6774,600657,信达地产,房地产,房地产开发和运营,2026-04-03,2026-04-07,3.00,2.94,-0.0200
6773,600231,凌钢股份,钢铁,普钢,2026-04-02,2026-04-03,2.06,2.03,-0.0146
6772,000002,万 科Ａ,房地产,房地产开发和运营,2026-04-02,2026-04-03,4.01,3.92,-0.0224
6745,601992,金隅集团,综合,综合Ⅱ,2026-04-01,2026-04-02,1.82,1.79,-0.0165
6752,601279,英利汽车,汽车,汽车零部件Ⅱ,2026-04-01,2026-04-02,4.30,4.19,-0.0256
6751,000590,古汉医药,医药,中药生产,2026-04-01,2026-04-02,11.88,11.96,0.0067
6750,000868,安凯客车,汽车,商用车,2026-04-01,2026-04-02,4.63,4.60,-0.0065
6749,000877,天山股份,建材,结构材料,2026-04-01,2026-04-02,5.06,5.03,-0.0059
6748,002678,珠江钢琴,轻工制造,文娱轻工Ⅱ,2026-04-01,2026-04-02,5.50,5.24,-0.0473
